# Track-B helper demo

**What this shows:** Shows how the Track-B slide-bag helper works on one tiny fixed example.

**Honest note:** This is a wiring check on **fake (synthetic) data**. It is **not** a scientific result. It uses no real patient data and never compares one group against another.

_Source: `scripts/trackb_demo.py` · Run after `pip install -e .`._


In [ ]:
"""Track B PoC demo — regenerate a synthetic subtype metrics.json into reports/EXP-fixture-trackb/.

Track B is HARD-GATED (LOCK-6, dormant until Track A clears G5). This produces NO real number:
it is PoC · synthetic · not patient-matched · gate-closed. The cohort is NOT patient-matched to
MRI; the ~84-patient Duke∩TCGA overlap MUST be de-duplicated before any "external" claim.

Reuses pinksight.eval.build_metrics_json (the single source of numbers) on the synthetic Track B
cohort fixture. Run:  PYTHONPATH=src .venv/bin/python scripts/trackb_demo.py
"""

from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "tests"))

from fixtures import trackb_synthetic as fx  
from pinksight.eval import build_metrics_json  

OUT = Path("reports/EXP-fixture-trackb")
_LABEL = "PoC · synthetic · not patient-matched · gate-closed"



In [ ]:
def main() -> None:
    cohort = fx.cohort_fixture()
    rng = np.random.default_rng(0)
    
    
    ki67 = {"y_true": rng.uniform(0, 60, fx.N_PATIENTS),
            "pred": rng.uniform(0, 60, fx.N_PATIENTS), "thresh": 14.0}

    doc = build_metrics_json(cohort, ki67, OUT, figures=True)

    
    
    
    
    doc.pop("ki67", None)
    doc["task"] = "subtype characterisation ONLY (Luminal-like vs TNBC)"

    
    doc["track"] = "B (TCGA WSI + genomics)"
    doc["status"] = _LABEL
    doc["gate"] = "HARD-GATED (LOCK-6 / decisions.md [5.1]) — dormant until Track A clears G5"
    doc["ki67_note"] = "Ki-67 block is a placeholder — Track B is SUBTYPE ONLY ([5.2]); ignore it."
    doc["ledger"] = "subtype characterisation ONLY; NO survival / growth-rate / early detection"
    doc["patient_match"] = ("NOT patient-matched to MRI; ~84-patient Duke∩TCGA overlap MUST be "
                            "de-duplicated before any external claim")
    doc["rung_meaning"] = fx.RUNG_MEANING  
    (OUT / "metrics.json").write_text(json.dumps(doc, indent=2, sort_keys=True) + "\n")

    top = doc["ablation_ladder"]["cross_attn"]["auroc"]
    print(f"[trackb_demo] {_LABEL}")  
    print(f"[trackb_demo] wrote {OUT/'metrics.json'}")  
    print(f"[trackb_demo] PoC subtype AUROC (wsi+genomics, synthetic): "  
          f"{top['value']} CI95 {top['ci95']}")



In [ ]:
if __name__ == "__main__":
    main()
